# Evaluation, transfer and limits: state of the art

**Level:** 3  
**Prerequisites:** Evaluation, transfer and limits level 1  
**Reading time:** 25 minutes

## Objectives

The learning objective is to explain the problem and its data, then inspect and, at higher levels, execute the released method on its matched task. A useful controller claim needs a valid denominator, independent outcomes, a fair baseline, disjoint evaluation and a task distribution where the baseline can fail. Many episodes from a few fixed templates do not create many independent situations.

## Setup

The versions below are pinned in `notebooks/requirements.txt` and are also recorded in this notebook's metadata.

```bash
pip install reflexmesh==0.1.2 numpy==2.4.6 matplotlib==3.11.2 torch==2.14.1 joblib==1.6.0 gymnasium==1.4.0 scikit-learn==1.9.1 stable-baselines3==2.9.2 sb3-contrib==2.9.2
```

## Data

The notebook reads committed product artifacts. Owner: Neuraxis. Repository: https://github.com/fsantibanezleal/CAOS_Neuraxis. License: Apache-2.0. The JSON is a first-party, frozen experiment artifact; no external benchmark data are redistributed in this notebook.

This notebook reads `data/derived/index.json` and the listed trace artifacts. The method run uses an isolated Reflexmesh case workspace. For learned methods, fetch the exact public release assets with `python -m reflexmesh.cli fetch-models --destination models/checkpoints --version v0.01.000`, then set `NEURAXIS_CHECKPOINTS` to that folder. The release manifest verifies asset hashes. Planner methods M09/M10 are read from frozen traces unless `NEURAXIS_NOTEBOOK_MODEL` explicitly names a configured local Ollama model.

Artifacts for this lesson:

- `data/derived/index.json`
- `data/derived/benchmark-adequacy.json`
- `data/derived/research.json`
- `data/derived/semantic-transfer/index.json`
- `data/derived/semantic-transfer/report.json`
- `data/derived/episodes/M01-C01-nominal-40000.json`
- `data/derived/episodes/M04-C01-nominal-40000.json`
- `data/derived/episodes/M09-C01-nominal-40000.json`
- `data/derived/episodes/M12-C01-nominal-40000.json`

## Method

Verify source digests, compare the core and semantic-transfer lanes separately, measure per-method outcomes and expose the paired transfer report. The package's independent outcome checks are authoritative; a successful teacher match, planner response or method label is not.

For a rate estimate, \(\hat p=\sum_i y_i/n\); the denominator includes failed, invalid and unresolved episodes according to the report contract. The Wilson interval in the core index is descriptive under an independent-episode assumption; clustered templates and identical seed outcomes reduce the effective information.

### Check the pinned analysis environment.

In [1]:
# Setup is performed outside the notebook; report the exact runtime pins.
import importlib.metadata
for distribution in ('reflexmesh', 'numpy', 'matplotlib', 'torch', 'joblib', 'gymnasium', 'scikit-learn', 'stable-baselines3', 'sb3-contrib'):
    print('{}=={}'.format(distribution, importlib.metadata.version(distribution)))

reflexmesh==0.1.2
numpy==2.4.6
matplotlib==3.11.2
torch==2.14.1
joblib==1.6.0
gymnasium==1.4.0
scikit-learn==1.9.1
stable-baselines3==2.9.2
sb3-contrib==2.9.2


### Verify the audit and each trace's bytes before interpreting outcomes.

In [2]:
# Read committed evidence and use the released API; never define another policy.
import hashlib, json
from pathlib import Path
ROOT = Path('../../../').resolve()
index_path = ROOT / 'data/derived/index.json'
index_bytes = index_path.read_bytes()
index = json.loads(index_bytes)
audit = json.loads((ROOT / 'data/derived/benchmark-adequacy.json').read_text(encoding='utf-8'))
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
selected_methods = ['M01', 'M04', 'M09', 'M12']
sample_case = 'C01'
seed = 40000
records = {}
for method_id in selected_methods:
    row = next(r for r in index['episodes'] if r['method_id'] == method_id and r['case_id'] == sample_case and r['variant'] == 'nominal' and r['seed'] == seed)
    trace_path = ROOT / 'data/derived' / row['path']
    trace_bytes = trace_path.read_bytes()
    assert hashlib.sha256(trace_bytes).hexdigest() == row['sha256']
    records[method_id] = json.loads(trace_bytes)
print('Index digest:', hashlib.sha256(index_bytes).hexdigest())
print('Evidence records:', len(records), '| case:', sample_case, '| seed:', seed)
print('Methods:', ', '.join(selected_methods))

Index digest: 1024a451fdc1f1aebc4e8d8d66e6bf1aca119f26ae058d3de9b84ff312d8d7a6
Evidence records: 4 | case: C01 | seed: 40000
Methods: M01, M04, M09, M12


### Inspect the original decisions and outcomes

In [3]:
# Summarize the recorded first action, authorized candidates, and independent oracle outcome.
for method_id, episode in records.items():
    first = episode['decisions'][0]
    allowed = [candidate['action'] for candidate in first['candidates'] if candidate['allowed']]
    print(method_id, '| action=', first['action'], '| reason=', first['reason'], '| allowed=', allowed)
    print('  success=', episode['metrics']['task_success'], '| steps=', episode['metrics']['steps'], '| violations=', episode['metrics']['violations'])

M01 | action= a-move | reason= explicit_state_guard | allowed= ['a-ack_event', 'a-move', 'a-refresh', 'a-inspect', 'a-index', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M04 | action= a-move | reason= calibrated_demonstration_preference | allowed= ['a-ack_event', 'a-move', 'a-refresh', 'a-inspect', 'a-index', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M09 | action= a-move | reason= The goal is to route the document to its destination; 'move' is the standard operation for transferring files between locations. | allowed= ['a-ack_event', 'a-move', 'a-refresh', 'a-inspect', 'a-index', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M12 | action= a-move | reason= persistent_outcome_aware_metacontrol | allowed= ['a-ack_event', 'a-move', 'a-refresh', 'a-inspect', 'a-index', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0


### Execute the released policies and compare the run

In [4]:
# Planner outcomes come from frozen traces; no LLM call is implicit.
import os
from pathlib import Path
from reflexmesh.environments import CaseSpec, SoftwareEnvironment
from reflexmesh.pipeline import make_policy
from reflexmesh.planners import OllamaPlanner
checkpoint_dir = Path(os.environ.get('NEURAXIS_CHECKPOINTS', str(ROOT / 'models/checkpoints')))
planner_model = os.environ.get('NEURAXIS_NOTEBOOK_MODEL')
reruns = {}
for method_id in selected_methods:
    if method_id in ('M09', 'M10') and not planner_model:
        print(method_id, 'not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above')
        continue
    planner = OllamaPlanner(planner_model) if planner_model and method_id in ('M09', 'M10', 'M11', 'M12') else None
    policy = make_policy(method_id, checkpoint_dir, planner=planner)
    policy.reset('neuraxis-notebook-' + method_id)
    with SoftwareEnvironment(CaseSpec(sample_case, 'nominal', seed)) as env:
        while not env.terminal:
            _, outcome = env.step(policy.predict(env.observe()))
        success = env.verify()
    recorded = records[method_id]
    reruns[method_id] = {'success': success, 'recorded_success': bool(recorded['metrics']['task_success']), 'recorded_first_action': recorded['decisions'][0]['action']}
    print(method_id, reruns[method_id])

M01 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-move'}
M04 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-move'}
M09 not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above
M12 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-move'}


### Compare the full sequence, not only the first action

In [5]:
# Sequence equality is diagnostic; the independent verifier determines task success.
for method_id, episode in records.items():
    actions = [decision['action'] for decision in episode['decisions']]
    print(method_id, ' -> '.join(actions), '| terminal=', episode['status'], '| verified=', episode['metrics']['task_success'])

M01 a-move | terminal= succeeded | verified= 1
M04 a-move | terminal= succeeded | verified= 1
M09 a-move | terminal= succeeded | verified= 1
M12 a-move | terminal= succeeded | verified= 1


### Recompute per-variant outcomes from every trace in the case

In [6]:
# Read every seed for the selected case and keep each failed episode in the denominator.
from collections import defaultdict
groups = defaultdict(list)
for row in index['episodes']:
    if row['case_id'] not in (sample_case,) or row['method_id'] not in selected_methods:
        continue
    raw = (ROOT / 'data/derived' / row['path']).read_bytes()
    assert hashlib.sha256(raw).hexdigest() == row['sha256']
    episode = json.loads(raw)
    groups[(row['method_id'], row['variant'])].append(int(bool(episode['metrics']['task_success'])))
for (method_id, variant), values in sorted(groups.items()):
    print(method_id, variant, 'successes=', sum(values), 'episodes=', len(values), 'rate=', round(sum(values)/len(values), 3))

M01 boundary successes= 10 episodes= 10 rate= 1.0
M01 delayed_observation successes= 10 episodes= 10 rate= 1.0
M01 duplicate_event successes= 10 episodes= 10 rate= 1.0
M01 nominal successes= 10 episodes= 10 rate= 1.0
M01 stale_conflict successes= 10 episodes= 10 rate= 1.0
M01 unavailable_dependency successes= 10 episodes= 10 rate= 1.0
M04 boundary successes= 10 episodes= 10 rate= 1.0
M04 delayed_observation successes= 10 episodes= 10 rate= 1.0
M04 duplicate_event successes= 10 episodes= 10 rate= 1.0
M04 nominal successes= 10 episodes= 10 rate= 1.0
M04 stale_conflict successes= 10 episodes= 10 rate= 1.0
M04 unavailable_dependency successes= 10 episodes= 10 rate= 1.0
M09 boundary successes= 10 episodes= 10 rate= 1.0
M09 delayed_observation successes= 9 episodes= 10 rate= 0.9
M09 duplicate_event successes= 10 episodes= 10 rate= 1.0
M09 nominal successes= 10 episodes= 10 rate= 1.0
M09 stale_conflict successes= 10 episodes= 10 rate= 1.0
M09 unavailable_dependency successes= 10 episodes= 10 

### Plot the variant stress without pooling families

In [7]:
# Each point summarizes the seeds from one case variant, not the full benchmark.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
variants = sorted(set(key[1] for key in groups))
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
for method_id in selected_methods:
    rates = [sum(groups[(method_id, variant)])/len(groups[(method_id, variant)]) if groups[(method_id, variant)] else float('nan') for variant in variants]
    ax.plot(variants, rates, marker='o', label=method_id)
ax.set_ylim(-0.05, 1.05)
ax.set_ylabel('Verified success rate')
ax.set_title(sample_case + ' by variant')
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=.25)
ax.legend()
display(fig)
plt.close(fig)

<Figure size 900x450 with 1 Axes>

### Keep semantic transfer in its own lane

In [8]:
# Verify the index and display transfer aggregates without pooling them with core.
transfer_path = ROOT / 'data/derived/semantic-transfer/index.json'
transfer_bytes = transfer_path.read_bytes()
transfer = json.loads(transfer_bytes)
transfer_report = json.loads((ROOT / 'data/derived/semantic-transfer/report.json').read_text(encoding='utf-8'))
assert hashlib.sha256(transfer_bytes).hexdigest() == transfer_report['index_sha256']
for row in transfer['aggregate_results']:
    print(row['method_id'], 'successes=', round(row['task_success'] * row['episodes']), 'episodes=', row['episodes'], 'rate=', row['task_success'])
print('transfer episodes=', transfer_report['episodes_checked'], '| families=', len(transfer['cases']))

M01 successes= 0 episodes= 128 rate= 0.0
M02 successes= 0 episodes= 128 rate= 0.0
M03 successes= 0 episodes= 128 rate= 0.0
M04 successes= 2 episodes= 128 rate= 0.015625
M05 successes= 0 episodes= 128 rate= 0.0
M06 successes= 1 episodes= 128 rate= 0.0078125
M07 successes= 0 episodes= 128 rate= 0.0
M08 successes= 6 episodes= 128 rate= 0.046875
M09 successes= 38 episodes= 128 rate= 0.296875
M10 successes= 0 episodes= 128 rate= 0.0
M11 successes= 0 episodes= 128 rate= 0.0
M12 successes= 18 episodes= 128 rate= 0.140625
transfer episodes= 1536 | families= 8


### Verify semantic traces and paired comparisons

In [9]:
# Re-read every semantic trace, verify its hash, and pair by case, variant, and seed.
semantic_pairs = {}
for row in transfer['episodes']:
    raw = (ROOT / 'data/derived/semantic-transfer' / row['path']).read_bytes()
    assert hashlib.sha256(raw).hexdigest() == row['sha256']
    outcome = int(bool(json.loads(raw)['metrics']['task_success']))
    semantic_pairs[(row['method_id'], row['case_id'], row['variant'], row['seed'])] = outcome
assert len(semantic_pairs) == transfer_report['episodes_checked'] == 1536
for method_id in sorted({key[0] for key in semantic_pairs}):
    vals = [value for key, value in semantic_pairs.items() if key[0] == method_id]
    print(method_id, 'successes=', sum(vals), '/', len(vals), 'rate=', round(sum(vals)/len(vals), 4))
for opponent in ('M04', 'M09'):
    paired = [(semantic_pairs[(opponent, case, variant, seed)], semantic_pairs[('M12', case, variant, seed)]) for _, case, variant, seed in semantic_pairs if _ == opponent]
    wins = sum(a > b for a, b in paired); losses = sum(a < b for a, b in paired)
    print('M12 vs', opponent, '| pairs=', len(paired), '| M12 wins=', losses, '| M12 losses=', wins, '| ties=', len(paired)-wins-losses)
print('Persisted M12 paired audit:', transfer_report['m12_paired'])

M01 successes= 0 / 128 rate= 0.0
M02 successes= 0 / 128 rate= 0.0
M03 successes= 0 / 128 rate= 0.0
M04 successes= 2 / 128 rate= 0.0156
M05 successes= 0 / 128 rate= 0.0
M06 successes= 1 / 128 rate= 0.0078
M07 successes= 0 / 128 rate= 0.0
M08 successes= 6 / 128 rate= 0.0469
M09 successes= 38 / 128 rate= 0.2969
M10 successes= 0 / 128 rate= 0.0
M11 successes= 0 / 128 rate= 0.0
M12 successes= 18 / 128 rate= 0.1406
M12 vs M04 | pairs= 128 | M12 wins= 18 | M12 losses= 2 | ties= 108
M12 vs M09 | pairs= 128 | M12 wins= 2 | M12 losses= 22 | ties= 104
Persisted M12 paired audit: {'M01': {'m12_wins': 18, 'ties': 110}, 'M04': {'m12_losses': 2, 'm12_wins': 18, 'ties': 108}, 'M09': {'m12_losses': 22, 'm12_wins': 2, 'ties': 104}}


### Recompute the complete core matrix and paired outcomes

In [10]:
# Verify per-episode SHA-256, recompute outcomes, and measure variation across seeds.
from collections import defaultdict
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
core_outcomes = {}
seed_variation = defaultdict(set)
for row in index['episodes']:
    raw = (ROOT / 'data/derived' / row['path']).read_bytes()
    assert hashlib.sha256(raw).hexdigest() == row['sha256']
    outcome = int(bool(json.loads(raw)['metrics']['task_success']))
    key = (row['method_id'], row['case_id'], row['variant'], row['seed'])
    core_outcomes[key] = outcome
    seed_variation[key[:3]].add(outcome)
assert len(core_outcomes) == 14400 and len(seed_variation) == 1440
print('Hash-verified episodes:', len(core_outcomes))
for row in sorted(index['aggregate_results'], key=lambda r: r['method_id']):
    print(row['method_id'], round(row['task_success']*row['episodes']), '/', row['episodes'], '| rate=', round(row['task_success'],4), '| decision p50/p95 ms=', round(row['controller_p50_ms'],4), round(row['controller_p95_ms'],4))
variable_cells = sum(len(values) > 1 for values in seed_variation.values())
assert variable_cells == audit['seed_variable_cells'] == 71
print('Seed-variable method/case/variant cells:', variable_cells, '/', len(seed_variation))
for left, right in [('M01','M12'), ('M04','M12'), ('M09','M12')]:
    pairs = [(core_outcomes[(left,c,v,s)], core_outcomes[(right,c,v,s)]) for _,c,v,s in core_outcomes if _ == left]
    wins = sum(a > b for a,b in pairs); losses = sum(a < b for a,b in pairs)
    print(left, 'vs', right, '| paired=',len(pairs),'| left wins=',wins,'| right wins=',losses,'| ties=',len(pairs)-wins-losses)
research = json.loads((ROOT / 'data/derived/research.json').read_text(encoding='utf-8'))
print('Structural transfer (separate 288-episode lane):')
for row in research['transfer']['methods']:
    print(row['method_id'], row['successes'], '/', row['episodes'], '| rate=', row['success_rate'])

Hash-verified episodes: 14400
M01 1200 / 1200 | rate= 1.0 | decision p50/p95 ms= 0.0219 0.0394
M02 1200 / 1200 | rate= 1.0 | decision p50/p95 ms= 0.0689 0.1177
M03 1200 / 1200 | rate= 1.0 | decision p50/p95 ms= 2.5695 4.2114
M04 1200 / 1200 | rate= 1.0 | decision p50/p95 ms= 2.6327 4.0172
M05 1200 / 1200 | rate= 1.0 | decision p50/p95 ms= 3.189 4.4434
M06 253 / 1200 | rate= 0.2108 | decision p50/p95 ms= 2.5673 4.0432
M07 1170 / 1200 | rate= 0.975 | decision p50/p95 ms= 2.3607 2.9987
M08 664 / 1200 | rate= 0.5533 | decision p50/p95 ms= 9.3875 14.0328
M09 595 / 1200 | rate= 0.4958 | decision p50/p95 ms= 1175.267 1718.18
M10 1200 / 1200 | rate= 1.0 | decision p50/p95 ms= 0.0641 0.1588
M11 1200 / 1200 | rate= 1.0 | decision p50/p95 ms= 1.8156 3.1422
M12 1160 / 1200 | rate= 0.9667 | decision p50/p95 ms= 18.1303 158.9701
Seed-variable method/case/variant cells: 71 / 1440
M01 vs M12 | paired= 1200 | left wins= 40 | right wins= 0 | ties= 1160
M04 vs M12 | paired= 1200 | left wins= 40 | right w

## State of the art

ToolSandbox and tau2 benchmark stateful tool interaction; SWE-bench evaluates repository changes with execution-based tests. These are complementary evaluation protocols, not interchangeable scores. Neuraxis's small designed transfer lane is useful for controlled stress but is not a full run of any external benchmark.

## Results

The core matrix is saturated: seven methods solve all 1,200 episodes and only 71/1,440 method/case/variant cells vary by seed. In semantic transfer, M04 solves 24/24 structural cases while M12 solves 21/24; the larger opaque semantic lane also favors M09 over M12. No learned-superiority claim is supported.

No generated outputs are committed by this authoring step; `scripts/run_notebooks.py` executes and stores them from a fresh kernel.

## Summary

The strongest result here is an honest diagnosis: the frozen core cannot answer the proposed superiority question. Transfer is bounded, separate, and presently favors simpler or alternative methods on key comparisons.

## References

- [TOOLSANDBOX](https://arxiv.org/abs/2408.04682)
- [SWE](https://arxiv.org/abs/2310.06770)
- [TAU2](https://arxiv.org/abs/2506.07982)